# Face Anonymization for Photogrammetry Scans

GDPR-compliant face removal pipeline for Einstar 3D photogrammetry scans.
Removes facial geometry while preserving all anatomical landmarks and optode positions.

**Pipeline:**
1. Load scan and detect nasion (auto via MediaPipe, or manual click)
2. Normalize axes, isolate head, detect landmarks, compute face mask + ear coverage
3. Preview the detected region
4. Interactive refinement: click on missed spots to add circular deletion zones
5. Delete facial vertices and visualize result

In [ ]:
import logging

import numpy as np
import pyvista as pv
import trimesh

import cedalion
import cedalion.io
import cedalion.plots
from cedalion.geometry.photogrammetry.anonymization import (
    detect_nasion_auto,
    pick_nasion,
    normalize_axes,
    isolate_head,
    detect_landmarks_from_nasion,
    get_facial_region_mask_from_nasion,
)
from cedalion.vtktutils import trimesh_to_vtk_polydata
from cedalion.dataclasses import VTKSurface
from scipy.spatial import KDTree

# Suppress verbose module and MediaPipe logging
logging.getLogger('cedalion').setLevel(logging.WARNING)
logging.getLogger('absl').setLevel(logging.WARNING)

pv.set_jupyter_backend('server')

# === CONFIG ===
SUBJECT_NUMBER = 1
SCANS_FOLDER = '/home/ma7/BA/PG_Subjects'  # set to your scan directory
FORCE_MANUAL = False       # set True to skip auto-detection and click nasion manually
PROXIMITY_RADIUS = 85.0    # mm — ear coverage band width
CIRCLE_RADIUS = 25.0       # mm — manual click-to-delete radius

## 1. Load scan + detect nasion

Auto-detect nasion via MediaPipe. If no face is found (or `FORCE_MANUAL=True`),
falls back to an interactive picker where the user clicks the nasion.

In [ ]:
path = f'{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}.obj'
surface = cedalion.io.read_einstar_obj(path)
print(f'Loaded: {surface.nvertices:,} vertices, {surface.nfaces:,} faces')

# Nasion detection
if not FORCE_MANUAL:
    auto_result = detect_nasion_auto(surface)
else:
    auto_result = None

if auto_result is not None:
    nasion, meta = auto_result
    print(f'Auto nasion: method={meta["method"]}, confidence={meta["confidence"]:.2f}')
else:
    print('Opening manual nasion picker...')
    nasion = pick_nasion(surface)
    if nasion is None:
        raise RuntimeError('No nasion selected — cannot proceed')
    meta = {'method': 'manual', 'confidence': 1.0}
    print(f'Manual nasion at: [{nasion[0]:.1f}, {nasion[1]:.1f}, {nasion[2]:.1f}]')

fwd = meta.get('forward_direction')
face_contour = meta.get('face_contour_3d')
print(f'Forward direction: {"available" if fwd is not None else "N/A"}')
print(f'Face contour: {len(face_contour)} pts' if face_contour is not None else 'Face contour: N/A')

## 2. Normalize axes, isolate head, detect landmarks, compute face mask + ear band

In [ ]:
# Normalize axes (X=up, Y=anterior, Z=left)
surface_n, nasion_n, R = normalize_axes(surface, nasion, forward_direction=fwd)

# Transform face contour and forward direction through rotation
if face_contour is not None:
    face_contour = (R @ face_contour.T).T
fwd_norm = R @ fwd if fwd is not None else np.array([0.0, 1.0, 0.0])

# Isolate head (remove shoulders/body)
surface_h, head_mask = isolate_head(surface_n, nasion_n)
print(f'Head isolation: {surface_n.nvertices:,} -> {surface_h.nvertices:,} vertices')

# Detect landmarks
landmarks = detect_landmarks_from_nasion(surface_h, nasion_n)
nz = landmarks.sel(label='Nz').pint.dequantify().values
lpa = landmarks.sel(label='LPA').pint.dequantify().values
rpa = landmarks.sel(label='RPA').pint.dequantify().values
for label in landmarks.label.values:
    pos = landmarks.sel(label=label).pint.dequantify().values
    print(f'  {label}: [{pos[0]:.1f}, {pos[1]:.1f}, {pos[2]:.1f}]')

# Face mask (spherical projection from MediaPipe contour, or hemisphere fallback)
face_mask = get_facial_region_mask_from_nasion(
    surface_h, nz, fwd_norm,
    face_contour_3d=face_contour,
    protected_points=landmarks,
)

# Ear coverage: proximity band around contour points
verts = surface_h.mesh.vertices

if face_contour is not None:
    forehead_pts = face_contour[face_contour[:, 0] > nasion_n[0]]
    x_upper = forehead_pts[:, 0].min() if len(forehead_pts) > 0 else nasion_n[0]
    y_posterior = min(lpa[1], rpa[1])

    contour_tree = KDTree(face_contour)
    dists, _ = contour_tree.query(verts, k=1)
    ear_band = (dists < PROXIMITY_RADIUS) & (verts[:, 0] <= x_upper) & (verts[:, 1] >= y_posterior)

    # Upward fill: extend band to x_upper using YZ proximity
    band_verts = verts[ear_band]
    if len(band_verts) > 0:
        band_tree_yz = KDTree(band_verts[:, 1:3])
        fill_dists, _ = band_tree_yz.query(verts[:, 1:3], k=1)
        fill_mask = (fill_dists < PROXIMITY_RADIUS) & (verts[:, 0] <= x_upper) & (verts[:, 0] >= verts[ear_band, 0].min()) & (verts[:, 1] >= y_posterior)
        ear_band = ear_band | fill_mask

    extended_mask = face_mask | ear_band
else:
    extended_mask = face_mask
    ear_band = np.zeros_like(face_mask)

band_only = extended_mask & ~face_mask
print(f'\nFace mask: {face_mask.sum():,} vertices')
print(f'Ear band:  {band_only.sum():,} vertices')
print(f'Total:     {extended_mask.sum():,} vertices ({100*extended_mask.sum()/len(extended_mask):.1f}%)')
if face_contour is not None:
    print(f'Posterior limit (LPA/RPA Y): {y_posterior:.1f}mm')

## 3. Preview anonymization region

In [ ]:
pvplt = pv.Plotter()
cedalion.plots.plot_surface(pvplt, surface_h, opacity=1.0)

# Deletion region (red)
if extended_mask.sum() > 0:
    pvplt.add_mesh(pv.PolyData(verts[extended_mask]), color='red', point_size=3,
                   opacity=0.4, render_points_as_spheres=True,
                   label=f'deletion region ({extended_mask.sum():,})')

# Face contour points (orange)
if face_contour is not None:
    for pt in face_contour:
        pvplt.add_mesh(pv.Sphere(radius=1.5, center=pt), color='orange')

# Landmarks (green)
for label in landmarks.label.values:
    pos = landmarks.sel(label=label).pint.dequantify().values
    pvplt.add_mesh(pv.Sphere(radius=3, center=pos), color='lime')
    pvplt.add_point_labels([pos], [label], font_size=16, point_size=0,
                           text_color='lime', shape=None, always_visible=True)

pvplt.add_text(
    f'S{SUBJECT_NUMBER} | deletion region: {extended_mask.sum():,} vertices',
    position='upper_left', font_size=12,
)
pvplt.add_legend()
pvplt.show()

## 4. Interactive refinement — click to add deletion circles

Left-click on any missed spot to add a circular deletion zone (radius = `CIRCLE_RADIUS`).
Each click adds a translucent sphere and updates the mask.
Close the window when done.

In [ ]:
refined_mask = extended_mask.copy()
click_count = [0]

# Protected zone: vertices near landmarks that cannot be added to mask
protected_positions = landmarks.pint.dequantify().values
protection_radius = 15.0  # mm

def on_pick(picked_point):
    if picked_point is None:
        return
    point = np.array(picked_point)

    # Check if inside protected zone
    dist_to_protected = np.linalg.norm(protected_positions - point, axis=1)
    if np.any(dist_to_protected < protection_radius):
        print('  Skipped — too close to a protected landmark')
        return

    # Find all vertices within CIRCLE_RADIUS
    dists = np.linalg.norm(verts - point, axis=1)
    circle = dists < CIRCLE_RADIUS
    new_verts = circle & ~refined_mask
    n_new = new_verts.sum()

    if n_new == 0:
        return

    # Add to mask
    refined_mask[new_verts] = True
    click_count[0] += 1

    # Visual feedback
    sphere = pv.Sphere(radius=CIRCLE_RADIUS, center=point)
    pvplt.add_mesh(sphere, color='yellow', opacity=0.3)

    # Update mask coloring on mesh (in-place scalar update)
    pv_mesh['mask'] = refined_mask.astype(float)

    print(f'  Click {click_count[0]}: +{n_new:,} vertices '
          f'(total: {refined_mask.sum():,})')

# Build plotter (non-notebook for interactive picking)
pvplt = pv.Plotter(notebook=False)

vtk_surface = VTKSurface.from_trimeshsurface(surface_h)
pv_mesh = pv.wrap(vtk_surface.mesh)

# Color mesh by mask: white=keep, red=delete
pv_mesh['mask'] = refined_mask.astype(float)
pvplt.add_mesh(pv_mesh, scalars='mask', cmap=['white', 'red'],
               clim=[0, 1], show_scalar_bar=False, opacity=0.9,
               smooth_shading=True, pickable=True)

# Show landmarks
for label in landmarks.label.values:
    pos = landmarks.sel(label=label).pint.dequantify().values
    pvplt.add_mesh(pv.Sphere(radius=3, center=pos), color='lime')
    pvplt.add_point_labels([pos], [label], font_size=12, point_size=0,
                           text_color='lime', shape=None, always_visible=True)

pvplt.enable_surface_point_picking(
    callback=on_pick,
    left_clicking=True,
    show_point=False,
    tolerance=0.005,
    show_message=f'Left-click to add deletion circle (r={CIRCLE_RADIUS}mm). Close when done.',
)

pvplt.add_text(
    f'S{SUBJECT_NUMBER} | Click to add deletion zones (r={CIRCLE_RADIUS}mm)\n'
    f'Green = protected landmarks | Close window when done',
    position='upper_left', font_size=10,
)
pvplt.show()

# After window closes
extended_mask = refined_mask
print(f'\nDone — {click_count[0]} clicks, final mask: {extended_mask.sum():,} vertices')

## 5. Anonymize — delete facial vertices

In [ ]:
# Delete faces where ANY vertex is in the mask
mesh_copy = surface_h.mesh.copy()
face_verts_in_mask = extended_mask[mesh_copy.faces]
faces_to_remove = face_verts_in_mask.any(axis=1)
mesh_copy.update_faces(~faces_to_remove)
mesh_copy.remove_unreferenced_vertices()

n_removed = surface_h.nvertices - len(mesh_copy.vertices)
print(f'Original:    {surface_h.nvertices:,} verts, {surface_h.nfaces:,} faces')
print(f'Anonymized:  {len(mesh_copy.vertices):,} verts, {len(mesh_copy.faces):,} faces')
print(f'Removed:     {n_removed:,} verts')

# Side-by-side visualization
anon_vtk_mesh = trimesh_to_vtk_polydata(mesh_copy)

pvplt = pv.Plotter(shape=(1, 2))

pvplt.subplot(0, 0)
cedalion.plots.plot_surface(pvplt, surface_h, opacity=1.0)
pvplt.add_text(f'S{SUBJECT_NUMBER} Original', position='upper_left', font_size=14)

pvplt.subplot(0, 1)
pvplt.add_mesh(pv.wrap(anon_vtk_mesh), rgb=True)
pvplt.add_text(f'S{SUBJECT_NUMBER} Anonymized (-{n_removed:,} verts)',
               position='upper_left', font_size=14)

pvplt.link_views()
pvplt.show()